In [7]:
# 1. Check the # & order of embeddings

import scanpy as sc
import numpy as np
import os
from sklearn.preprocessing import StandardScaler


In [8]:
team_files = {
    'scGPT': 'scGPT_results.h5ad',
    'PCA & scVI': 'PCA_and_scVI_results.h5ad', 
    'C2S': 'C2S_results.h5ad'
}

keys = {
    'scGPT': 'X_scGPT',
    'PCA & scVI': ['X_pca', 'X_scvi', 'X_scvi30'],
    'C2S': 'X_C2S'
}

In [9]:
def merge_h5ad_embeddings(file_dict, key_dict):
    all_embeds = []
    reference_barcodes = None
    
    print("🚀 Start merging...")

    for model_name, path in file_dict.items():
        if not os.path.exists(path):
            print(f"🚨 No file : {model_name} ({path})")
            return None
        
        # read h5ad
        adata = sc.read_h5ad(path)
        print(f"📦 {model_name} load done : {adata.shape}")
        
        # Check the barcode order 
        current_barcodes = adata.obs_names.values
        if reference_barcodes is None:
            reference_barcodes = current_barcodes
            print(f"✅ reference barcode setting done ({model_name})")
        else:
            if not np.array_equal(reference_barcodes, current_barcodes):
                print(f"❌ Error : cell order of {model_name} is different with reference.")
                return None
        
        # Extract embedding from obsm
        target_keys = key_dict[model_name]

        if isinstance(target_keys, str):
            target_keys = [target_keys]
            
        for k in target_keys:
            if k in adata.obsm:
                all_embeds.append(adata.obsm[k])
                print(f"   - '{k}' Extraction complete: {adata.obsm[k].shape}")
            else:
                print(f"🚨 Error : No '{k}' key in {model_name} file.")
                return None

    # Feature Fusion
    fused_matrix = np.concatenate(all_embeds, axis=1)
    print(f"\n🔗 Merging complete ! Final dimension : {fused_matrix.shape}")
    
    return fused_matrix, reference_barcodes

In [10]:
result = merge_h5ad_embeddings(team_files, keys)

🚀 Start merging...
📦 scGPT load done : (180794, 2000)
✅ reference barcode setting done (scGPT)
   - 'X_scGPT' Extraction complete: (180794, 512)
📦 PCA & scVI load done : (180794, 1)
   - 'X_pca' Extraction complete: (180794, 50)
   - 'X_scvi' Extraction complete: (180794, 10)
   - 'X_scvi30' Extraction complete: (180794, 30)
📦 C2S load done : (180794, 24719)
   - 'X_C2S' Extraction complete: (180794, 1024)

🔗 Merging complete ! Final dimension : (180794, 1626)


In [11]:
import anndata
anndata.settings.allow_write_nullable_strings = True

if result:
    fused_matrix, barcodes = result
    
    # Scaling
    # # scaler = StandardScaler()
    # # scaled_embedding = scaler.fit_transform(fused_matrix)
    # print("⚖️ Scaling done.")

    # Save
    np.save('scaled_embedding.npy', fused_matrix)
    final_adata = sc.AnnData(X=fused_matrix, obs=dict(obs_names=barcodes))
    final_adata.write_h5ad('RNA_combined_final.h5ad')
    print("💾 Final file saved : scaled_embedding.npy, RNA_combined_final.h5ad")

💾 Final file saved : scaled_embedding.npy, RNA_combined_final.h5ad


In [ ]:
import h5py

h5_file_path = 'RNA_embeddings_final.h5'

with h5py.File(h5_file_path, 'w') as f:
    # embeddings
    f.create_dataset('embeddings', data=scaled_embedding)
    
    # cell_ids
    ascii_barcodes = [b.encode("utf-8") for b in barcodes]
    f.create_dataset('cell_ids', data=ascii_barcodes)

print(f"✅ Final file saved : {h5_file_path}")

In [ ]:
import h5py
import numpy as np

h5_file_path = 'RNA_embeddings_final.h5'

with h5py.File(h5_file_path, 'r') as f:
    print("📂 H5 file keys :", list(f.keys()))
    
    embed_shape = f['embeddings'].shape
    id_count = f['cell_ids'].shape[0]
    print(f"📊 Embedding size : {embed_shape}")
    print(f"🆔 barcode number : {id_count}")

    target_barcode = f['cell_ids'][0].decode('utf-8')
    print(f"\n🔍 specific barcode we want to check : {target_barcode}")
    
    # How to search
    all_ids = [x.decode('utf-8') for x in f['cell_ids'][:]]
    if target_barcode in all_ids:
        idx = all_ids.index(target_barcode)
        target_vector = f['embeddings'][idx]
        print(f"✅ Search done ! first 5 of index {idx} vector :")
        print(target_vector[:5])